# 02 — Party System Analysis
## Tamil Nadu Assembly Elections (1971–2021)

**Objectives:**
- Party dominance timeline (seat share per election)
- Vote share vs seat share disproportionality (Gallagher index)
- Effective Number of Parties (ENOP) trend
- Alliance structure inference
- Minor party lifecycle tracking

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from db import query, query_all, GENERAL_ELECTION_YEARS, MAJOR_PARTIES, ALLIANCES

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

df = query_all()
ge = df[df.year.isin(GENERAL_ELECTION_YEARS)].copy()
winners = ge[ge.position == 1].copy()
print(f'General election records: {len(ge):,}, Winners: {len(winners):,}')

## 1. Party Dominance Timeline

In [ ]:
# Seats won by party per election
top_parties = ['DMK', 'ADMK', 'INC', 'ADK', 'PMK', 'DMDK', 'BJP', 'CPM', 'CPI', 'MDMK', 'NTK']

seat_counts = winners.groupby(['year', 'party']).size().unstack(fill_value=0)
# Consolidate into major + Other
other_cols = [c for c in seat_counts.columns if c not in top_parties]
seat_counts['Other'] = seat_counts[other_cols].sum(axis=1)
seat_counts = seat_counts[['DMK', 'ADMK', 'INC', 'ADK', 'PMK', 'DMDK', 'BJP', 'CPM', 'CPI', 'MDMK', 'NTK', 'Other']]
seat_counts = seat_counts.loc[seat_counts.index.isin(GENERAL_ELECTION_YEARS)]

# Party colors
party_colors = {
    'DMK': '#E53935', 'ADMK': '#43A047', 'INC': '#1E88E5', 'ADK': '#8E24AA',
    'PMK': '#F9A825', 'DMDK': '#00ACC1', 'BJP': '#FF6F00', 'CPM': '#C62828',
    'CPI': '#D32F2F', 'MDMK': '#6D4C41', 'NTK': '#00695C', 'Other': '#9E9E9E'
}

fig, ax = plt.subplots(figsize=(18, 8))
seat_counts.plot(kind='bar', stacked=True, ax=ax, 
                 color=[party_colors.get(c, '#9E9E9E') for c in seat_counts.columns],
                 edgecolor='white', linewidth=0.5)
ax.set_title('Seats Won by Party — Tamil Nadu General Elections (1971–2021)', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('Seats Won')
ax.legend(title='Party', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.axhline(y=117, color='black', linestyle='--', alpha=0.3, label='Majority (117/234)')
plt.tight_layout()
plt.savefig('/app/output/02_party_seats_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Seat share as percentage — stacked area
total_seats = seat_counts.sum(axis=1)
seat_pct = seat_counts.div(total_seats, axis=0) * 100

fig, ax = plt.subplots(figsize=(16, 8))
seat_pct[['DMK', 'ADMK', 'INC', 'Other']].plot(kind='area', stacked=True, ax=ax,
    color=['#E53935', '#43A047', '#1E88E5', '#9E9E9E'], alpha=0.7)
ax.set_title('Seat Share % Over Time — DMK vs ADMK vs INC vs Others', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('Seat Share (%)')
ax.set_ylim(0, 100)
ax.legend(title='Party')
plt.tight_layout()
plt.savefig('/app/output/02_seat_share_area.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Vote Share vs Seat Share (Disproportionality)

In [ ]:
# Calculate vote share and seat share for each party per election
def gallagher_index(year_data):
    """Calculate Gallagher index of disproportionality."""
    total_seats = year_data[year_data.position == 1].shape[0]
    total_votes = year_data.drop_duplicates(subset=['constituency_no']).valid_votes.sum()
    
    party_votes = year_data.groupby('party').votes.sum()
    party_seats = year_data[year_data.position == 1].groupby('party').size()
    
    all_parties = party_votes.index.union(party_seats.index)
    vote_share = (party_votes.reindex(all_parties, fill_value=0) / party_votes.sum() * 100)
    seat_share = (party_seats.reindex(all_parties, fill_value=0) / total_seats * 100)
    
    return np.sqrt(0.5 * ((vote_share - seat_share) ** 2).sum())


gallagher = []
for year in GENERAL_ELECTION_YEARS:
    yr_data = ge[ge.year == year]
    gi = gallagher_index(yr_data)
    gallagher.append({'year': year, 'gallagher_index': gi})

gal_df = pd.DataFrame(gallagher)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(gal_df.year, gal_df.gallagher_index, color='#7E57C2', edgecolor='white')
ax.set_title('Gallagher Index of Disproportionality — TN Elections', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('Gallagher Index (higher = more disproportional)')
for _, row in gal_df.iterrows():
    ax.annotate(f'{row.gallagher_index:.1f}', (row.year, row.gallagher_index),
               textcoords='offset points', xytext=(0, 5), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/app/output/02_gallagher_index.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Vote share vs seat share for DMK and ADMK
vs_ss = []
for year in GENERAL_ELECTION_YEARS:
    yr = ge[ge.year == year]
    total_votes = yr.votes.sum()
    total_seats = yr[yr.position == 1].shape[0]
    for party in ['DMK', 'ADMK']:
        p_votes = yr[yr.party == party].votes.sum()
        p_seats = yr[(yr.party == party) & (yr.position == 1)].shape[0]
        vs_ss.append({
            'year': year, 'party': party,
            'vote_share': p_votes / total_votes * 100 if total_votes > 0 else 0,
            'seat_share': p_seats / total_seats * 100 if total_seats > 0 else 0
        })

vs_df = pd.DataFrame(vs_ss)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for i, party in enumerate(['DMK', 'ADMK']):
    p = vs_df[vs_df.party == party]
    axes[i].plot(p.year, p.vote_share, 'o-', label='Vote Share %', color='#1E88E5', linewidth=2)
    axes[i].plot(p.year, p.seat_share, 's-', label='Seat Share %', color='#E53935', linewidth=2)
    axes[i].fill_between(p.year, p.vote_share, p.seat_share, alpha=0.1, color='gray')
    axes[i].set_title(f'{party}: Vote Share vs Seat Share')
    axes[i].set_xlabel('Election Year')
    axes[i].set_ylabel('%')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('FPTP Amplification Effect — DMK vs ADMK', fontsize=14)
plt.tight_layout()
plt.savefig('/app/output/02_vote_vs_seat_share.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Effective Number of Parties (ENOP) Trend

In [ ]:
# ENOP trend — average per constituency per election
enop_by_year = ge.drop_duplicates(subset=['year', 'constituency_no']).groupby('year').agg(
    avg_enop=('enop', 'mean'),
    median_enop=('enop', 'median'),
    std_enop=('enop', 'std')
).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(enop_by_year.year, enop_by_year.avg_enop, 'o-', color='#FF7043', linewidth=2, markersize=8)
ax.fill_between(enop_by_year.year, 
                enop_by_year.avg_enop - enop_by_year.std_enop,
                enop_by_year.avg_enop + enop_by_year.std_enop, alpha=0.15, color='#FF7043')
ax.set_title('Effective Number of Parties (ENOP) — Average per Constituency', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('ENOP')
ax.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='ENOP=2 (pure two-party)')
ax.legend()

for _, row in enop_by_year.iterrows():
    ax.annotate(f'{row.avg_enop:.2f}', (row.year, row.avg_enop),
               textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/app/output/02_enop_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nENOP Interpretation:')
print('< 2.0: Dominant party system')
print('2.0-3.0: Two-party / two-coalition system')
print('> 3.0: Multi-party fragmentation')

## 4. Alliance Structure Inference

In [ ]:
# For each election with defined alliances, calculate alliance-level performance
alliance_results = []
for year, blocs in ALLIANCES.items():
    yr_winners = winners[winners.year == year]
    yr_all = ge[ge.year == year]
    total_seats = len(yr_winners)
    total_votes = yr_all.votes.sum()
    
    for bloc_name, parties in blocs.items():
        bloc_seats = yr_winners[yr_winners.party.isin(parties)].shape[0]
        bloc_votes = yr_all[yr_all.party.isin(parties)].votes.sum()
        alliance_results.append({
            'year': year, 'alliance': bloc_name,
            'seats': bloc_seats,
            'seat_share': bloc_seats / total_seats * 100,
            'vote_share': bloc_votes / total_votes * 100
        })

alliances_df = pd.DataFrame(alliance_results)

fig = px.bar(alliances_df, x='year', y='seats', color='alliance', barmode='group',
             text='seats', title='Alliance-Level Seat Performance (2001–2021)',
             color_discrete_map={'DMK+': '#E53935', 'ADMK+': '#43A047', 'PMK+': '#F9A825',
                                  'DMDK+': '#00ACC1', 'NTK': '#00695C', 'MNM': '#7E57C2'})
fig.update_layout(height=500)
fig.show()

In [ ]:
# Alliance vote share vs seat share
fig = px.scatter(alliances_df, x='vote_share', y='seat_share', color='alliance',
                 size='seats', text='year', title='Alliance Vote Share vs Seat Share',
                 labels={'vote_share': 'Vote Share (%)', 'seat_share': 'Seat Share (%)'},
                 color_discrete_map={'DMK+': '#E53935', 'ADMK+': '#43A047', 'PMK+': '#F9A825',
                                      'DMDK+': '#00ACC1', 'NTK': '#00695C', 'MNM': '#7E57C2'})
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines', name='Proportional',
                          line=dict(dash='dash', color='gray')))
fig.update_layout(height=500)
fig.show()

## 5. Minor Party Lifecycle

In [ ]:
# Track rise and fall of notable minor parties
minor_parties = ['DMDK', 'NTK', 'PMK', 'MDMK', 'BJP']

minor_data = ge[ge.party.isin(minor_parties)].groupby(['year', 'party']).agg(
    candidates=('candidate', 'count'),
    total_votes=('votes', 'sum'),
    avg_vote_share=('vote_share_percentage', 'mean'),
    seats_won=('position', lambda x: (x == 1).sum())
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, metric, title in zip(axes.flatten(), 
    ['candidates', 'total_votes', 'avg_vote_share', 'seats_won'],
    ['Candidates Fielded', 'Total Votes', 'Avg Vote Share (%)', 'Seats Won']):
    for party in minor_parties:
        p = minor_data[minor_data.party == party]
        ax.plot(p.year, p[metric], 'o-', label=party, 
                color=party_colors.get(party, '#999'), linewidth=2, markersize=6)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Election Year')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Minor Party Lifecycle — TN Elections', fontsize=14)
plt.tight_layout()
plt.savefig('/app/output/02_minor_party_lifecycle.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary
print('=== PARTY SYSTEM ANALYSIS SUMMARY ===')
print(f'\nDMK total seats won (all elections): {winners[winners.party=="DMK"].shape[0]}')
print(f'ADMK total seats won: {winners[winners.party=="ADMK"].shape[0]}')
print(f'INC total seats won: {winners[winners.party=="INC"].shape[0]}')
print(f'\nGallagher Index range: {gal_df.gallagher_index.min():.1f} – {gal_df.gallagher_index.max():.1f}')
print(f'Mean ENOP (all elections): {enop_by_year.avg_enop.mean():.2f}')
print(f'\nParties that won seats: {winners.party.nunique()}')
print(f'Parties that contested: {ge.party.nunique()}')